# 01 — EDA gates and four-way chronological split (issue S1-06)

**Purpose.** (1) Answer the EDA gates G1–G7 from `docs/FEATURE_SPEC.md` section 7 with real numbers. (2) Cut PaySim into train / validation / test / replay pool in chronological order and save the boundaries to `docs/split_boundaries.json`.

**Discipline built into this notebook**
- Gates that need **no label** (G1, G3, G4, G5) run on all rows.
- Everything that looks at **`isFraud`** (G2, G7 and the exploration section) runs on the **train split only**, so no feature-design decision is informed by validation, test or replay labels. G6 only *counts* rows and frauds per split and reads no patterns from them.
- The split uses **no randomness** (it is chronological), so it is the same on every run. `RANDOM_STATE` is recorded for the later notebooks.
- The cutoffs in the next cell are the **pre-registration**: change them only **before** the first run, commit, then run.

**Run order:** restart the kernel, run all cells. Run twice; `docs/split_boundaries.json` must come out identical.

In [8]:
from pathlib import Path

p = Path(r"D:\Projects\fraud_detection\data\cache\paysim.parquet")
print(p)
print(p.exists())

D:\Projects\fraud_detection\data\cache\paysim.parquet
True


In [11]:
import sys, json, hashlib
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # plain-python fallback
    display = print

# --- locate the repo root so imports work from notebooks/ or from the root ---
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "notebooks"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from data_loader import load_paysim
from shared.features import CONFIG            # window length comes from the ONE shared config

# ------------------------------------------------------------------
# PRE-REGISTRATION: edit BEFORE the first run, commit, then run.
# ------------------------------------------------------------------
RANDOM_STATE = 42
SCOPED_TYPES = ["TRANSFER", "CASH_OUT"]
WINDOW_STEPS = CONFIG.window_steps            # 24 (1 step = 1 hour in PaySim)
SPLIT_FRACTIONS = {"train": 0.70, "validation": 0.15, "test": 0.10, "replay": 0.05}
G1_MATERIAL_SHARE = 0.05      # G1: origin accounts "repeat materially" if >= 5% of scoped rows have a repeated origin
G2_MIN_FRAUD_SHARE = 0.05     # G2: chain "carries signal" only if >= 5% of fraud cash-outs have a prior inbound transfer ...
G2_MIN_LIFT = 2.0             # ... AND that share is >= 2x the share among legitimate cash-outs
MIN_FRAUD_PER_SPLIT = 50      # warn if a split has fewer fraud rows than this (assert > 0 always)
EXPECTED_ROWS = 6_362_620     # from the README / data_loader run
EXPECTED_COLS = ["step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig",
                 "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud"]
print("repo root:", ROOT, "| window steps:", WINDOW_STEPS)

repo root: d:\Projects\fraud_detection | window steps: 24


## 1. Load and integrity checks

In [12]:
import inspect
import notebooks.data_loader as loader

print(loader.__file__)
print(inspect.signature(loader.load_paysim))
print(loader.DEFAULT_PARQUET_PATH)
print(loader.DEFAULT_PARQUET_PATH.exists())

d:\Projects\fraud_detection\notebooks\data_loader.py
(csv_path: pathlib.Path = WindowsPath('data/raw/paysim.csv'), parquet_path: pathlib.Path = WindowsPath('D:/Projects/fraud_detection/data/cache/paysim.parquet'), force_rebuild: bool = False) -> pandas.DataFrame
D:\Projects\fraud_detection\data\cache\paysim.parquet
True


In [13]:
from pathlib import Path

parquet = Path(r"D:\Projects\fraud_detection\data\cache\paysim.parquet")

df = load_paysim(
    parquet_path=parquet,
    force_rebuild=False,
)

print(df.shape)

(6362620, 11)


In [15]:
import inspect

print(inspect.getsource(load_paysim))

def load_paysim(
    csv_path: Path = DEFAULT_CSV_PATH,
    parquet_path: Path = DEFAULT_PARQUET_PATH,
    force_rebuild: bool = False,
) -> pd.DataFrame:
    """
    Load the PaySim dataset, using a local Parquet cache when available.

    Parameters
    ----------
    csv_path : Path
        Path to the original PaySim CSV, as downloaded from Kaggle
        (https://www.kaggle.com/datasets/ealaxi/paysim1).
    parquet_path : Path
        Path where the Parquet cache is stored/read.
    force_rebuild : bool
        If True, ignore any existing Parquet cache and rebuild it
        from the CSV (use this if the raw CSV changes).

    Returns
    -------
    pd.DataFrame
        The loaded PaySim dataset.

    Raises
    ------
    FileNotFoundError
        If no Parquet cache exists and the raw CSV is not found either.
    """
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    if parquet_path.exists() and not force_rebuild:
        logger.info("Loading PaySim from Parquet c

In [18]:
df = load_paysim(
    parquet_path=Path(r"D:\Projects\fraud_detection\data\cache\paysim.parquet"),
    force_rebuild=False,
)
print(df.shape)
assert set(df.columns) == set(EXPECTED_COLS), f"unexpected columns: {list(df.columns)}"
assert isinstance(df.index, pd.RangeIndex) and df.index.start == 0 and df.index.step == 1, \
    "index must be 0..n-1 in original CSV order (row position is the clock)"
assert int(df.isna().sum().sum()) == 0, "nulls found"

n = len(df)
print(f"rows: {n:,} | columns: {df.shape[1]}", "" if n == EXPECTED_ROWS else f"  <-- expected {EXPECTED_ROWS:,}")
display(df.dtypes.to_frame("dtype").T)

type_table = df.groupby("type", observed=True).agg(rows=("isFraud", "size"), fraud=("isFraud", "sum"))
display(type_table)
assert set(df.loc[df.isFraud == 1, "type"].astype(str)) <= set(SCOPED_TYPES), \
    "fraud found outside TRANSFER/CASH_OUT: the scope statement is wrong, STOP"
print("OK: every fraud row is a TRANSFER or CASH_OUT (documented fact confirmed).")

(6362620, 11)
rows: 6,362,620 | columns: 11 


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
dtype,int64,category,float64,str,float64,float64,str,float64,float64,int8,int8


,rows,fraud
type,,
CASH_IN,1399284,0
CASH_OUT,2237500,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,532909,4097


OK: every fraud row is a TRANSFER or CASH_OUT (documented fact confirmed).


## 2. Label-free gates (all rows): G3, G4, G5, G1

In [19]:
# G3 — chronological order. The whole design depends on this.
g3_monotonic = bool(df["step"].is_monotonic_increasing)    # True means non-decreasing
assert g3_monotonic, "G3 FAILED: step is not non-decreasing in row order"
print(f"G3 PASS: step non-decreasing | min {df.step.min()}, max {df.step.max()}, distinct steps {df.step.nunique()}")

scoped_mask = df["type"].isin(SCOPED_TYPES)

# G4 — merchants as recipients of scoped types
g4_merchant_dest_scoped = int(df.loc[scoped_mask, "nameDest"].str.startswith("M").sum())
g4_merchant_orig_all = int(df["nameOrig"].str.startswith("M").sum())
print(f"G4: merchant ('M...') recipients in TRANSFER/CASH_OUT = {g4_merchant_dest_scoped:,} | merchant origins (all rows) = {g4_merchant_orig_all:,}")
print("    ->", "no dest_is_merchant feature needed" if g4_merchant_dest_scoped == 0 else "ADD dest_is_merchant feature (spec G4 rule)")

# G5 — self transfers
g5_self_all = int((df["nameOrig"] == df["nameDest"]).sum())
g5_self_scoped = int((df.loc[scoped_mask, "nameOrig"] == df.loc[scoped_mask, "nameDest"]).sum())
print(f"G5: nameOrig == nameDest: all rows {g5_self_all:,}, scoped rows {g5_self_scoped:,} (function already handles it)")

G3 PASS: step non-decreasing | min 1, max 743, distinct steps 743
G4: merchant ('M...') recipients in TRANSFER/CASH_OUT = 0 | merchant origins (all rows) = 0
    -> no dest_is_merchant feature needed
G5: nameOrig == nameDest: all rows 0, scoped rows 0 (function already handles it)


In [20]:
# G1 — do origin accounts repeat?
vc = df["nameOrig"].value_counts()
n_unique_orig = int(len(vc))
distinct_share = n_unique_orig / n
rows_repeated_all = int(vc[vc > 1].sum()) / n
scoped_counts = df.loc[scoped_mask, "nameOrig"].map(vc)
g1_scoped_repeat_share = float((scoped_counts > 1).mean())
print(f"G1: distinct nameOrig = {n_unique_orig:,} of {n:,} rows ({distinct_share:.4%} distinct)")
print(f"    rows whose origin appears more than once: all rows {rows_repeated_all:.4%}, scoped rows {g1_scoped_repeat_share:.4%}")
print("    (a third-party notebook reported 6,353,307 distinct origins; compare, do not assume)")
g1_material = g1_scoped_repeat_share >= G1_MATERIAL_SHARE
print("    cutoff", f"{G1_MATERIAL_SHARE:.0%}", "->", "REPEATS MATERIALLY: revisit per-origin history features (0.2.0)" if g1_material else "D4 STANDS: per-origin history stays out of 0.1.0")

# G1b (information): do recipients repeat? Features #15/#16 depend on it.
transfers = df.loc[df["type"] == "TRANSFER", "nameDest"]
dvc = transfers.value_counts()
g1b_transfer_dest_repeat_share = float(transfers.map(dvc).gt(1).mean())
print(f"G1b: share of TRANSFER rows whose recipient receives more than one TRANSFER overall: {g1b_transfer_dest_repeat_share:.4%}")
del vc, scoped_counts, transfers, dvc

G1: distinct nameOrig = 6,353,307 of 6,362,620 rows (99.8536% distinct)
    rows whose origin appears more than once: all rows 0.2925%, scoped rows 0.2930%
    (a third-party notebook reported 6,353,307 distinct origins; compare, do not assume)
    cutoff 5% -> D4 STANDS: per-origin history stays out of 0.1.0
G1b: share of TRANSFER rows whose recipient receives more than one TRANSFER overall: 77.4795%


## 3. The four-way chronological split

Cut by **row position** in the original order, but each cut is **snapped to the start of a `step`**, so no step is shared between two splits. Order: train, then validation, then test, then replay pool (the most recent ~5%). No randomness is used.

In [21]:
step_arr = df["step"].to_numpy()

def snap_to_step_start(row_idx: int) -> int:
    row_idx = min(max(row_idx, 1), n - 1)
    return int(np.searchsorted(step_arr, step_arr[row_idx], side="left"))

names = ["train", "validation", "test", "replay"]
assert abs(sum(SPLIT_FRACTIONS.values()) - 1.0) < 1e-9
cuts, cum = [], 0.0
for name in names[:-1]:
    cum += SPLIT_FRACTIONS[name]
    cuts.append(snap_to_step_start(int(round(cum * n))))
bounds = [0] + cuts + [n]
assert all(a < b for a, b in zip(bounds, bounds[1:])), f"degenerate split bounds {bounds}"

SPLITS = {name: (bounds[i], bounds[i + 1]) for i, name in enumerate(names)}   # [start, end) row positions
parts = {name: df.iloc[s:e] for name, (s, e) in SPLITS.items()}

# --- assertions the issue card requires ---
assert sum(len(p) for p in parts.values()) == n, "rows lost or duplicated"
assert SPLITS["train"][0] == 0 and SPLITS["replay"][1] == n
for a, b in zip(names, names[1:]):
    assert SPLITS[a][1] == SPLITS[b][0], f"{a} and {b} are not contiguous"
    assert parts[a]["step"].max() < parts[b]["step"].min(), f"{a} and {b} share a step"
assert not (set(parts["train"].index) & set(parts["test"].index))
print("split assertions passed: contiguous, ordered, no shared steps, every row in exactly one split")

split assertions passed: contiguous, ordered, no shared steps, every row in exactly one split


In [22]:
# G6 — base rates and BOTH denominators, per split (counts only; no pattern is read from validation/test/replay)
rows = []
for name, p in parts.items():
    sc = p["type"].isin(SCOPED_TYPES)
    fraud = int(p["isFraud"].sum())
    rows.append({
        "split": name, "row_from": SPLITS[name][0], "row_to_excl": SPLITS[name][1],
        "step_min": int(p["step"].min()), "step_max": int(p["step"].max()),
        "rows": len(p), "share_of_rows": len(p) / n,
        "scoped_rows": int(sc.sum()), "fraud": fraud,
        "fraud_rate_all_rows": fraud / len(p),
        "fraud_rate_scoped_rows": fraud / int(sc.sum()),
    })
split_table = pd.DataFrame(rows).set_index("split")
display(split_table.assign(share_of_rows=split_table["share_of_rows"].round(4),
                           fraud_rate_all_rows=split_table["fraud_rate_all_rows"].round(6),
                           fraud_rate_scoped_rows=split_table["fraud_rate_scoped_rows"].round(6)))

for name in names:
    f = int(split_table.loc[name, "fraud"])
    assert f > 0, f"{name} split has no fraud rows: boundaries need rethinking"
    if f < MIN_FRAUD_PER_SPLIT:
        print(f"WARNING: {name} split has only {f} fraud rows (< {MIN_FRAUD_PER_SPLIT}); metrics from it will be noisy")

tot_f = int(df.isFraud.sum())
print(f"\nG6 overall: fraud {tot_f:,} | rate over all rows {tot_f / n:.4%} | rate over scoped rows {tot_f / int(scoped_mask.sum()):.4%}")
print("Every reported metric must state which denominator it uses.")

,row_from,row_to_excl,step_min,step_max,rows,share_of_rows,scoped_rows,fraud,fraud_rate_all_rows,fraud_rate_scoped_rows
split,,,,,,,,,,
train,0,4433703,1,322,4433703,0.6968,1938484,3633,0.000819,0.001874
validation,4433703,5406876,323,377,973173,0.1530,423848,570,0.000586,0.001345
test,5406876,6040750,378,489,633874,0.0996,274283,1246,0.001966,0.004543
replay,6040750,6362620,490,743,321870,0.0506,133794,2764,0.008587,0.020659



G6 overall: fraud 8,213 | rate over all rows 0.1291% | rate over scoped rows 0.2965%
Every reported metric must state which denominator it uses.


## 4. Label-conditioned gates — TRAIN split only: G2 and G7

In [23]:
train = parts["train"]

# G2 — does the transfer-then-cash-out chain exist at the ACCOUNT level?
# For every CASH_OUT in train: did its sender receive a TRANSFER earlier (earlier row) within the window?
tr = train.loc[train["type"] == "TRANSFER", ["step", "nameDest"]].copy()
tr["t_row"] = tr.index
co = train.loc[train["type"] == "CASH_OUT", ["step", "nameOrig", "isFraud"]].copy()
co["c_row"] = co.index

common = set(tr["nameDest"]) & set(co["nameOrig"])
print(f"G2: accounts that are both a TRANSFER recipient and a CASH_OUT sender in train: {len(common):,}")

tr_c = tr[tr["nameDest"].isin(common)]
co_c = co[co["nameOrig"].isin(common)]
pairs = co_c.merge(tr_c, left_on="nameOrig", right_on="nameDest", suffixes=("_co", "_tr"))
pairs = pairs[pairs["t_row"] < pairs["c_row"]]                      # strictly earlier row
in_window = pairs[(pairs["step_co"] - pairs["step_tr"]) < WINDOW_STEPS]

co["chain_ever"] = co["c_row"].isin(pairs["c_row"])
co["chain_window"] = co["c_row"].isin(in_window["c_row"])

g2 = co.groupby("isFraud").agg(cash_outs=("c_row", "size"),
                               chain_ever=("chain_ever", "mean"),
                               chain_window=("chain_window", "mean"))
g2.index = g2.index.map({0: "legitimate", 1: "fraud"})
display(g2)

fraud_share = float(g2.loc["fraud", "chain_window"]) if "fraud" in g2.index else float("nan")
legit_share = float(g2.loc["legitimate", "chain_window"])
lift = (fraud_share / legit_share) if legit_share > 0 else (float("inf") if fraud_share > 0 else float("nan"))
g2_signal = bool(fraud_share >= G2_MIN_FRAUD_SHARE and lift >= G2_MIN_LIFT)
print(f"fraud cash-outs with an inbound transfer in the last {WINDOW_STEPS} steps: {fraud_share:.4%}")
print(f"legitimate cash-outs with the same:                                  {legit_share:.4%}")
print(f"lift = {lift:.2f} | cutoffs: share >= {G2_MIN_FRAUD_SHARE:.0%} and lift >= {G2_MIN_LIFT}")
print("->", "CHAIN CARRIES SIGNAL: keep features 11-14" if g2_signal else
      "CHAIN DOES NOT CARRY SIGNAL: drop features 11-14; do NOT claim FIU red flag 3 as operationalised")

# Reverse view (information): do recipients of fraud TRANSFERs ever cash out later (in train)?
co_senders = set(co["nameOrig"])
tr_all = train.loc[train["type"] == "TRANSFER", ["nameDest", "isFraud"]]
rev = tr_all.assign(dest_cashes_out=tr_all["nameDest"].isin(co_senders)).groupby("isFraud")["dest_cashes_out"].mean()
rev.index = rev.index.map({0: "legitimate TRANSFER", 1: "fraud TRANSFER"})
print("\nshare of TRANSFER recipients that appear as a CASH_OUT sender somewhere in train:")
display(rev.to_frame("share"))

G2: accounts that are both a TRANSFER recipient and a CASH_OUT sender in train: 131


,cash_outs,chain_ever,chain_window
isFraud,,,
legitimate,1569816,0.00004,0.000011
fraud,1826,0.00000,0.000000


fraud cash-outs with an inbound transfer in the last 24 steps: 0.0000%
legitimate cash-outs with the same:                                  0.0011%
lift = 0.00 | cutoffs: share >= 5% and lift >= 2.0
-> CHAIN DOES NOT CARRY SIGNAL: drop features 11-14; do NOT claim FIU red flag 3 as operationalised

share of TRANSFER recipients that appear as a CASH_OUT sender somewhere in train:


,share
isFraud,
legitimate TRANSFER,0.000825
fraud TRANSFER,0.000000


In [24]:
# G7 — isFlaggedFraud: what it is, and how R1 behaves on TRAIN
flag_all = int(df["isFlaggedFraud"].sum())
flagged = df[df["isFlaggedFraud"] == 1]
print(f"G7: isFlaggedFraud == 1 in {flag_all:,} rows (all rows; this is a data-documentation check, no label pattern read)")
if flag_all:
    print("    types:", flagged["type"].astype(str).value_counts().to_dict(), "| min amount:", f"{flagged['amount'].min():,.2f}")
    g7_all_transfers = bool((flagged["type"].astype(str) == "TRANSFER").all())
    g7_all_above = bool((flagged["amount"] > 200_000).all())
    print(f"    all TRANSFER: {g7_all_transfers} | all amount > 200,000: {g7_all_above}")
    tr_flag = train[train["type"] == "TRANSFER"]
    big = tr_flag[tr_flag["amount"] > 200_000]
    print(f"    in train: TRANSFERs above 200,000 = {len(big):,}; of those flagged = {int(big['isFlaggedFraud'].sum()):,}"
          " (documented rule vs actual behaviour)")
else:
    g7_all_transfers = g7_all_above = None

sc_train = train[train["type"].isin(SCOPED_TYPES)]
tp = int(((sc_train["isFlaggedFraud"] == 1) & (sc_train["isFraud"] == 1)).sum())
fp = int(((sc_train["isFlaggedFraud"] == 1) & (sc_train["isFraud"] == 0)).sum())
fn = int(((sc_train["isFlaggedFraud"] == 0) & (sc_train["isFraud"] == 1)).sum())
r1_prec = tp / (tp + fp) if tp + fp else float("nan")
r1_rec = tp / (tp + fn) if tp + fn else float("nan")
print(f"R1 on TRAIN (scoped rows): TP {tp}, FP {fp}, FN {fn} -> precision {r1_prec:.3f}, recall {r1_rec:.4f}")

G7: isFlaggedFraud == 1 in 16 rows (all rows; this is a data-documentation check, no label pattern read)
    types: {'TRANSFER': 16} | min amount: 353,874.22
    all TRANSFER: True | all amount > 200,000: True
    in train: TRANSFERs above 200,000 = 278,905; of those flagged = 3 (documented rule vs actual behaviour)
R1 on TRAIN (scoped rows): TP 3, FP 0, FN 3630 -> precision 1.000, recall 0.0008


## 5. Train-only exploration (informational)

R2 and R3 are **already fixed** in `docs/FEATURE_SPEC.md` section 6.1 and are not tuned here. This section only shows how the fraud/legitimate rows look in the **train** split, to sanity-check the "drain" and "bulk" hypotheses and the balance-consistency group.

In [25]:
sc = train[train["type"].isin(SCOPED_TYPES)].copy()
sc["label"] = sc["isFraud"].map({0: "legitimate", 1: "fraud"})
sc["drained"] = (sc["oldbalanceOrg"] > 0) & (sc["newbalanceOrig"] == 0)
sc["bulk_200k"] = sc["amount"] > 200_000
sc["orig_err_nonzero"] = (sc["oldbalanceOrg"] - sc["amount"] - sc["newbalanceOrig"]).abs() > 1e-6
sc["dest_err_nonzero"] = (sc["oldbalanceDest"] + sc["amount"] - sc["newbalanceDest"]).abs() > 1e-6
sc["dest_old_zero"] = sc["oldbalanceDest"] == 0
sc["dest_new_zero"] = sc["newbalanceDest"] == 0
sc["is_transfer"] = sc["type"] == "TRANSFER"

shares = sc.groupby("label")[["is_transfer", "drained", "bulk_200k", "orig_err_nonzero", "dest_err_nonzero",
                               "dest_old_zero", "dest_new_zero"]].mean().T
shares["lift_fraud_over_legit"] = shares["fraud"] / shares["legitimate"].replace(0, np.nan)
print("share of rows with each property, TRAIN scoped rows:")
display(shares)

q = sc.groupby("label")["amount"].quantile([0.5, 0.9, 0.99]).unstack()
print("amount quantiles by class (train, scoped):")
display(q)

# Preview of R2 / R3 on TRAIN only (definitions fixed in the spec)
for label, rule in [("R2 drained", sc["drained"]), ("R3 drained & > 200,000", sc["drained"] & sc["bulk_200k"])]:
    tp = int((rule & (sc.isFraud == 1)).sum()); fp = int((rule & (sc.isFraud == 0)).sum()); fn = int((~rule & (sc.isFraud == 1)).sum())
    pr = tp / (tp + fp) if tp + fp else float("nan"); rc = tp / (tp + fn) if tp + fn else float("nan")
    print(f"{label:<24} TP {tp:>6} FP {fp:>8} FN {fn:>6} | precision {pr:.3f} recall {rc:.3f}")
print("(train only. The report numbers for R1-R3 come from the TEST split in S1-11.)")

share of rows with each property, TRAIN scoped rows:


label,fraud,legitimate,lift_fraud_over_legit
is_transfer,0.497385,0.188663,2.636366
drained,0.979356,0.423156,2.314408
bulk_200k,0.652353,0.430578,1.515063
orig_err_nonzero,0.008808,0.907619,0.009705
dest_err_nonzero,0.609689,0.312563,1.950614
dest_old_zero,0.645747,0.135985,4.748679
dest_new_zero,0.494357,0.006042,81.815728


amount quantiles by class (train, scoped):


,0.50,0.90,0.99
label,,,
fraud,417217.89,3924102.18,1.000000e+07
legitimate,170744.26,526527.46,2.187363e+06


R2 drained               TP   3558 FP   818744 FN     75 | precision 0.004 recall 0.979
R3 drained & > 200,000   TP   2309 FP   417409 FN   1324 | precision 0.006 recall 0.636
(train only. The report numbers for R1-R3 come from the TEST split in S1-11.)


## 6. Save the split boundaries and print the gate summary

In [26]:
fingerprint = hashlib.sha256(
    f"{n}|{int(df['step'].sum())}|{df['amount'].sum():.2f}|{int(df['isFraud'].sum())}".encode()).hexdigest()

payload = {
    "description": "Chronological 4-way split of PaySim by row position in original CSV order; cuts snapped to step starts. No randomness.",
    "random_state_for_later_notebooks": RANDOM_STATE,
    "n_rows": n,
    "dataset_fingerprint_sha256": fingerprint,
    "split_fractions_requested": SPLIT_FRACTIONS,
    "splits": {
        name: {
            "row_start": SPLITS[name][0], "row_end_exclusive": SPLITS[name][1],
            "step_min": int(split_table.loc[name, "step_min"]), "step_max": int(split_table.loc[name, "step_max"]),
            "rows": int(split_table.loc[name, "rows"]),
            "scoped_rows": int(split_table.loc[name, "scoped_rows"]),
            "fraud": int(split_table.loc[name, "fraud"]),
        } for name in names
    },
}

out_path = ROOT / "docs" / "split_boundaries.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
if out_path.exists():
    old = json.loads(out_path.read_text(encoding="utf-8"))
    if old == payload:
        print("split_boundaries.json: IDENTICAL to the committed file (reproducible)")
    else:
        print("!!! split_boundaries.json: DIFFERS from the committed file. Dataset or settings changed. Review before committing.")
out_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print("written:", out_path)

written: d:\Projects\fraud_detection\docs\split_boundaries.json


In [27]:
fmt = lambda x: "n/a" if x is None else f"{x:.4%}"
print("Paste into docs/FEATURE_SPEC.md section 9 (Gate results):\n")
print(f"- G1: distinct nameOrig {n_unique_orig:,}/{n:,} ({distinct_share:.4%}); scoped rows with repeated origin {g1_scoped_repeat_share:.4%} (cutoff {G1_MATERIAL_SHARE:.0%}) -> {'revisit per-origin history' if g1_material else 'D4 stands'}")
print(f"- G2 (train): fraud cash-outs with inbound transfer <= {WINDOW_STEPS} steps {fraud_share:.4%} vs legitimate {legit_share:.4%}, lift {lift:.2f} -> {'keep features 11-14' if g2_signal else 'drop features 11-14'}")
print(f"- G3: step non-decreasing = {g3_monotonic}")
print(f"- G4: merchant recipients in scoped types = {g4_merchant_dest_scoped:,} -> {'no change' if g4_merchant_dest_scoped == 0 else 'add dest_is_merchant'}")
print(f"- G5: self-transfers: all {g5_self_all:,}, scoped {g5_self_scoped:,}")
print(f"- G6: fraud {tot_f:,}; rate all rows {tot_f / n:.4%}; rate scoped rows {tot_f / int(scoped_mask.sum()):.4%}")
print(f"- G7: isFlaggedFraud=1 rows {flag_all:,}; R1 on train precision {r1_prec:.3f}, recall {r1_rec:.4f}")
print("\nSplit rows:", {k: v['rows'] for k, v in payload['splits'].items()})

Paste into docs/FEATURE_SPEC.md section 9 (Gate results):

- G1: distinct nameOrig 6,353,307/6,362,620 (99.8536%); scoped rows with repeated origin 0.2930% (cutoff 5%) -> D4 stands
- G2 (train): fraud cash-outs with inbound transfer <= 24 steps 0.0000% vs legitimate 0.0011%, lift 0.00 -> drop features 11-14
- G3: step non-decreasing = True
- G4: merchant recipients in scoped types = 0 -> no change
- G5: self-transfers: all 0, scoped 0
- G6: fraud 8,213; rate all rows 0.1291%; rate scoped rows 0.2965%
- G7: isFlaggedFraud=1 rows 16; R1 on train precision 1.000, recall 0.0008

Split rows: {'train': 4433703, 'validation': 973173, 'test': 633874, 'replay': 321870}


## 7. Decision log (fill in by hand after reading the outputs)

| Gate | Result | Decision | Changes needed |
|---|---|---|---|
| G1 | | | |
| G2 | | | |
| G3 | | | |
| G4 | | | |
| G5 | | | |
| G6 | | | |
| G7 | | | |

If G2 says "drop features 11-14": edit `FEATURE_NAMES` and `compute_features` in `shared/features/core.py` **and** `docs/FEATURE_SPEC.md` in one PR, bump `FEATURE_SCHEMA_VERSION`, and rerun `pytest`.